In [1]:
import numpy as np
import scipy.special as sysp
import scipy.spatial as syspat
import sys
import pandas as pd
import gc
import matplotlib.pyplot as plt

In [2]:
class HODFits(object):
    def __init__(self):
        self.bf = {'lgMin':np.array([12.33,-0.85,0.19]),
                   'siglgM': np.array([0.44, -0.16,0.3]),
                   'lgM1':np.array([13.52,-0.72,0.16]),
                   'lgM0':np.array([12.24,-0.54,0.0]),
                   'alpha':np.array([1.16,-0.20,0.10])}

    def hod_fit(self, param, Mr):
        a0, a1, a2 = self.bf[param]
        x = Mr + 20.5
        out = (a0 + a1*x + a2*x**2 if param != 'siglgM' else  a0 + a1*sysp.erf(x/a2) )
        return out

    def hod_fit_deriv(self, param, Mr):
        a0, a1, a2 = self.bf[param]
        x = Mr + 20.5
        out = (a1 + 2*a2*x if param != 'siglgM' else (a1/a2)*2/np.sqrt(np.pi)*np.exp(-(x/a2)**2) )
        return out

In [3]:
class HODFunctions(HODFits):
    
    def __init__(self):
        HODFits.__init__(self)

    def erf_arg(self,Mr,lgm):
        Mr_sc = np.isscalar(Mr)
        lgm_sc = np.isscalar(lgm)

        lgm_arr = np.outer(lgm,np.ones(Mr.size)) if ((not Mr_sc) & (not lgm_sc)) else 1.0*lgm
        erfarg = (lgm_arr - self.hod_fit('lgMin', Mr))/self.hod_fit('siglgM',Mr)
        return erfarg

    def fcen_thresh(self,Mr,lgm):
        erfarg = self.erf_arg(Mr, lgm)
        out = 0.5*(1+sysp.erf(erfarg) )
        return out

    def Nsat_thresh(self, Mr, lgm):
        M0 = 10**self.hod_fit('lgM0',Mr)
        M1 = 10**self.hod_fit('lgM1',Mr)
        alpha = self.hod_fit('alpha',Mr)
        mhalo = 10**lgm
        Mr_sc = np.isscalar(Mr)
        lgm_sc = np.isscalar(lgm)

        if Mr_sc & lgm_sc:
            out = ((mhalo - M0)/M1)**alpha if mhalo > M0 else 0.0
        else:
            mhalo_arr = np.outer(mhalo,np.ones(Mr.size)) if (not Mr_sc) & (not lgm_sc) else 1.0*mhalo 
            Dm = (mhalo_arr - M0) / M1
            Dm[Dm <0.0] = 0.0 
            out = Dm**alpha
    
        return out
    

In [4]:
class Mocker(HODFunctions):
    def __init__(self):
        HODFunctions.__init__(self)
        self.halo_cat = 'out_1.parents'
        self.redshift = 0.0
        self.Om = 0.276
        self.Lbox = 300.0 # Mpc/h
        self.massdef = 'm200b'
        self.mmin = 2e11 # Msun/h
        self.Mrmax = -20.5
        self.Dvir = 200*self.Om
        
        self.rhoc = 2.7754e11 # (Msun/h) / (Mpc/h)^3
        self.rhoc_z = self.rhoc*self.Ehub(self.redshift)**2
        
        self.rng = np.random.RandomState(seed=42)

        dMr = 0.001
        nMr = int((23.5+self.Mrmax)/dMr)
        self.Mrvals = np.linspace(-23.5,self.Mrmax,nMr) # useful for interpolation later
        self.dMr = self.Mrvals[1]-self.Mrvals[0]
        
        self.halodatatype = {'ID':int,'descID':int,
                            'mbnd_vir':float,'vmax':float,'vrms':float,'rvir':float,'rs':float,'np':int,
                            'x':float,'y':float,'z':float,'vx':float,'vy':float,'vz':float,
                            'Jx':float,'Jy':float,'Jz':float,'spin':float,'rs_klypin':float,
                            'mvir':float,'m200b':float,'m200c':float,'mCustom2':float,'mCustom':float,
                            'Xoff':float,'Voff':float,'spin_bullock':float,'b_to_a':float,'c_to_a':float,
                            'Ax':float,'Ay':float,'Az':float,'b_to_a_500c':float,'c_to_a_500c':float,
                            'Ax_500c':float,'Ay_500c':float,'Az_500c':float,
                            'TbyU':float,'Mpe_Behroozi':float,'Mpe_Diemer':float,'halfmassradius':float,
                            'pid':int}

        # Check for parameter naming convention (lgMin vs lgMmin)
        lgMmin_param = 'lgMin' if 'lgMin' in self.bf else 'lgMmin'

        # Precompute HOD parameters for self.Mrvals and self.Mrmax
        self.lgMmin_Mrvals = self.hod_fit(lgMmin_param, self.Mrvals)
        self.siglgM_Mrvals = self.hod_fit('siglgM', self.Mrvals)
        self.M0_Mrvals = 10**self.hod_fit('lgM0', self.Mrvals)
        self.M1_Mrvals = 10**self.hod_fit('lgM1', self.Mrvals)
        self.alpha_Mrvals = self.hod_fit('alpha', self.Mrvals)

        self.lgMmin_Mrmax = self.hod_fit(lgMmin_param, self.Mrmax)
        self.siglgM_Mrmax = self.hod_fit('siglgM', self.Mrmax)
        self.M0_Mrmax = 10**self.hod_fit('lgM0', self.Mrmax)
        self.M1_Mrmax = 10**self.hod_fit('lgM1', self.Mrmax)
        self.alpha_Mrmax = self.hod_fit('alpha', self.Mrmax)
        
    def Ehub(self,z):
        return np.sqrt(self.Om*(1+z)**3 + 1-self.Om)
    
    def read_this(self):
        print(f"Attempting to read file from: {self.halo_cat}")
        try:
            df = pd.read_csv(self.halo_cat, 
                             dtype=self.halodatatype, 
                             names=list(self.halodatatype.keys()), 
                             comment='#', 
                             sep=r'\s+', 
                             header=None)
            print(f"Successfully loaded {len(df)} rows.")
            return df.to_records(index=False)
        except Exception as e:
            print(f"CRITICAL ERROR reading file: {e}")
            return None

    def prep_halo_data(self):
        halos = self.read_this()
        Nhalos_all = halos.size
        cond_clean = (halos[self.massdef] >= self.mmin)
        cond_clean = cond_clean & (halos['pid'] == -1)
        halos = halos[cond_clean]
        print(f"Kept {halos.size} of {Nhalos_all} objects in catalog")
        del cond_clean
        gc.collect()
        return halos

    def erf_arg(self, Mr, lgm):
        lgMmin_param = 'lgMin' if 'lgMin' in self.bf else 'lgMmin'
        if Mr is self.Mrvals:
            lgMmin = self.lgMmin_Mrvals
            siglgM = self.siglgM_Mrvals
            lgm_arr = lgm[:, np.newaxis]
        elif Mr == self.Mrmax:
            lgMmin = self.lgMmin_Mrmax
            siglgM = self.siglgM_Mrmax
            lgm_arr = lgm
        else:
            Mr_sc = np.isscalar(Mr)
            lgm_sc = np.isscalar(lgm)
            lgm_arr = lgm[:, np.newaxis] if ((not Mr_sc) and (not lgm_sc)) else 1.0 * lgm
            lgMmin = self.hod_fit(lgMmin_param, Mr)
            siglgM = self.hod_fit('siglgM', Mr)
        return (lgm_arr - lgMmin) / siglgM

    def fcen_thresh(self, Mr, lgm):
        erfarg = self.erf_arg(Mr, lgm)
        out = 0.5 * (1 + sysp.erf(erfarg))
        return out

    def Nsat_thresh(self, Mr, lgm):
        if Mr is self.Mrvals:
            M0 = self.M0_Mrvals
            M1 = self.M1_Mrvals
            alpha = self.alpha_Mrvals
            mhalo_arr = (10**lgm)[:, np.newaxis]
            Dm = (mhalo_arr - M0) / M1
            Dm[Dm < 0.0] = 0.0
            return Dm**alpha
        elif Mr == self.Mrmax:
            M0 = self.M0_Mrmax
            M1 = self.M1_Mrmax
            alpha = self.alpha_Mrmax
            mhalo = 10**lgm
            if np.isscalar(lgm):
                return ((mhalo - M0)/M1)**alpha if mhalo > M0 else 0.0
            else:
                Dm = (mhalo - M0) / M1
                Dm[Dm < 0.0] = 0.0
                return Dm**alpha
        else:
            M0 = 10**self.hod_fit('lgM0', Mr)
            M1 = 10**self.hod_fit('lgM1', Mr)
            alpha = self.hod_fit('alpha', Mr)
            mhalo = 10**lgm
            Mr_sc = np.isscalar(Mr)
            lgm_sc = np.isscalar(lgm)
            if Mr_sc and lgm_sc:
                out = ((mhalo - M0)/M1)**alpha if mhalo > M0 else 0.0
            else:
                mhalo_arr = mhalo[:, np.newaxis] if ((not Mr_sc) and (not lgm_sc)) else 1.0 * mhalo
                Dm = (mhalo_arr - M0) / M1
                Dm[Dm < 0.0] = 0.0
                out = Dm**alpha
            return out

    def occupy_halos(self, halo_mass):
        print("Occupying halos...")
        occupy = np.ones(halo_mass.size, dtype=bool)
        fcen_min = self.fcen_thresh(self.Mrmax, np.log10(halo_mass))
        u = self.rng.rand(halo_mass.size)
        occupy[u > fcen_min] = False
        print_string = f"... {np.where(occupy)[0].size} of {occupy.size} halos occupied\n"
        print_string += "--------------------------------"
        print(print_string)
        return occupy

    def assign_centrals(self, halos):
        print("Assigning values for centrals... ")
        centrals = np.zeros(halos.size, dtype=[('haloid','int'),
                                              ('lgm','double'),('rvir','double'),('con','double'),
                                              ('x','double'),('y','double'),('z','double'),
                                              ('Mr','double'),('Nsat','int')])
        print("... halo IDs, halo masses, positions")
        centrals['haloid'] = halos['ID']
        centrals['lgm'] = np.log10(halos[self.massdef])
        centrals['x'] = halos['x']
        centrals['y'] = halos['y']
        centrals['z'] = halos['z']
        
        rvir = (3*halos[self.massdef]/(4*np.pi*self.Dvir*self.rhoc_z))**(1/3.)
        centrals['rvir'] = rvir*(1+self.redshift)
        centrals['con'] = rvir/(halos['rs']*1e-3/(1+self.redshift))
        
        centrals['Mr'] = self.assign_central_luminosity(centrals['lgm'])
        centrals['Nsat'] = self.get_Poisson(centrals['lgm'])
        
        print_string = '... done\n'
        print_string += "--------------------------------"
        print(print_string)
        return centrals

    def get_Poisson(self, lgm_halos):
        print("... Poisson satellite counts")
        mean_Nsat = self.Nsat_thresh(self.Mrmax, lgm_halos)
        return self.rng.poisson(mean_Nsat)

    def assign_central_luminosity(self, lgm_halos, chunk_size=5000):
        print("... luminosities")
        Mr_cen = np.zeros(lgm_halos.size)
        u = self.rng.rand(lgm_halos.size)
        
        for i in range(0, lgm_halos.size, chunk_size):
            end = min(i + chunk_size, lgm_halos.size)
            lgm_chunk = lgm_halos[i:end]
            u_chunk = u[i:end]
            
            PcenL = self.fcen_thresh(self.Mrvals, lgm_chunk) / self.fcen_thresh(self.Mrmax, lgm_chunk)[:, np.newaxis]
            
            mask_gt = PcenL > u_chunk[:, np.newaxis]
            idx_upp = np.argmax(mask_gt, axis=1)
            Mr_upp = self.Mrvals[idx_upp]
            
            mask_le = PcenL <= u_chunk[:, np.newaxis]
            idx_low_reversed = np.argmax(mask_le[:, ::-1], axis=1)
            idx_low = PcenL.shape[1] - 1 - idx_low_reversed
            Mr_low = self.Mrvals[idx_low]
            
            any_le = np.any(mask_le, axis=1)
            Mr_low[~any_le] = -np.inf
            
            Mr_cen[i:end] = np.maximum(Mr_upp, Mr_low)

        return Mr_cen

    def assign_satellites(self, centrals):
        print("Assigning values for satellites... ")
        idx = np.repeat(np.arange(centrals.size), centrals['Nsat'])
        Nsat_tot = idx.size
        
        satellites = np.zeros(Nsat_tot, dtype=[('haloid','int'),
                                              ('lgm','double'),('con','double'),
                                              ('x','double'),('y','double'),('z','double'),
                                              ('Mr','double')])
        
        if Nsat_tot == 0:
            return satellites
            
        print("... halo properties")
        satellites['haloid'] = centrals['haloid'][idx]
        satellites['lgm'] = centrals['lgm'][idx]
        satellites['con'] = centrals['con'][idx]
        
        print("... positions")
        sat_pos = self.gen_NFW_profile(Nsat_tot, centrals['con'][idx], centrals['rvir'][idx])
        satellites['x'] = (sat_pos[:,0] + centrals['x'][idx]) % self.Lbox
        satellites['y'] = (sat_pos[:,1] + centrals['y'][idx]) % self.Lbox
        satellites['z'] = (sat_pos[:,2] + centrals['z'][idx]) % self.Lbox
        
        print("... luminosities")
        satellites['Mr'] = self.assign_satellite_luminosities(centrals['lgm'][idx])
        
        gc.collect()
        print_string = '... done\n'
        print_string += "--------------------------------"
        print(print_string)
        return satellites

    def assign_satellite_luminosities(self, lgm_sat, chunk_size=5000):
        Mr_sat = np.zeros(lgm_sat.size)
        u = self.rng.rand(lgm_sat.size)
        
        for i in range(0, lgm_sat.size, chunk_size):
            end = min(i + chunk_size, lgm_sat.size)
            lgm_chunk = lgm_sat[i:end]
            u_chunk = u[i:end]
            
            PsatL = self.Nsat_thresh(self.Mrvals, lgm_chunk) / self.Nsat_thresh(self.Mrmax, lgm_chunk)[:, np.newaxis]
            mask_ge = PsatL >= u_chunk[:, np.newaxis]
            idx_upp = np.argmax(mask_ge, axis=1)
            
            Mr_sat[i:end] = self.Mrvals[idx_upp]

        return Mr_sat

    def gen_NFW_profile(self, Nsat_tot, cvir_arr, Rvir_arr):
        if Nsat_tot == 0:
            return np.zeros((0,3))

        phi = 2 * np.pi * self.rng.rand(Nsat_tot)
        cos_theta = 2 * self.rng.rand(Nsat_tot) - 1
        sin_theta = np.sqrt(1 - cos_theta**2)
        vran = self.rng.rand(Nsat_tot)
        
        max_c = np.max(cvir_arr)
        x_grid = np.logspace(-5, np.log10(max_c * 2.0), 100000)
        M_grid = np.log(1 + x_grid) - x_grid / (1 + x_grid)
        
        target_M = vran * (np.log(1 + cvir_arr) - cvir_arr / (1 + cvir_arr))
        x_samp = np.interp(target_M, M_grid, x_grid)
        rsamp = x_samp * (Rvir_arr / cvir_arr)
        
        x_trc = rsamp * sin_theta * np.sin(phi)
        y_trc = rsamp * sin_theta * np.cos(phi)
        z_trc = rsamp * cos_theta
        
        return np.column_stack((x_trc, y_trc, z_trc))

    def mock_it(self):
        halos_all = self.prep_halo_data()
        occupy = self.occupy_halos(halos_all[self.massdef])
        halos = halos_all[occupy]
        all_masses = halos_all[self.massdef]
        del halos_all
        gc.collect()

        centrals = self.assign_centrals(halos)
        satellites = self.assign_satellites(centrals)
        del halos
        gc.collect()

        print_string = f"... {centrals.size + satellites.size} galaxies created\n"
        print_string += "--------------------------------"
        print(print_string)
        print_string = '... all done\n'
        print_string += "--------------------------------"
        print(print_string)
        return centrals, satellites, all_masses


In [5]:
# 1. Instantiate the God of the Simulation
mm = Mocker()

# 2. Run the full pipeline
# This will trigger the halo reading, HOD occupation, and galaxy placement
centrals, satellites, all_masses = mm.mock_it()

# 3. Quick Sanity Check
print(f"Total Central Galaxies: {centrals.size}")
print(f"Total Satellite Galaxies: {satellites.size}")

Attempting to read file from: out_1.parents
Successfully loaded 2941105 rows.
Kept 458324 of 2941105 objects in catalog
Occupying halos...
... 67102 of 458324 halos occupied
--------------------------------
Assigning values for centrals... 
... halo IDs, halo masses, positions
... luminosities
... Poisson satellite counts
... done
--------------------------------
Assigning values for satellites... 
... halo properties
... positions
... luminosities
... done
--------------------------------
... 83375 galaxies created
--------------------------------
... all done
--------------------------------
Total Central Galaxies: 67102
Total Satellite Galaxies: 16273


Kept 458324 of 2941105 objects in catalog
Occupying halos...
... 67102 of 458324 halos occupied
--------------------------------
Assigning values for centrals... 
... halo IDs, halo masses, positions
... luminosities


... Poisson satellite counts
... done
--------------------------------
Assigning values for satellites... 
... halo properties
... positions
... luminosities


... done
--------------------------------
... 83375 galaxies created
--------------------------------
... all done
--------------------------------
Total Central Galaxies: 67102
Total Satellite Galaxies: 16273


In [6]:
import time

# Create the simulator object
mm = Mocker()

# Time the full execution
start_time = time.time()
centrals, satellites, allx_masses = mm.mock_it()
end_time = time.time()

print(f"Simulation completed in {end_time - start_time:.2f} seconds.")

Attempting to read file from: out_1.parents
Successfully loaded 2941105 rows.
Kept 458324 of 2941105 objects in catalog
Occupying halos...
... 67102 of 458324 halos occupied
--------------------------------
Assigning values for centrals... 
... halo IDs, halo masses, positions
... luminosities
... Poisson satellite counts
... done
--------------------------------
Assigning values for satellites... 
... halo properties
... positions
... luminosities
... done
--------------------------------
... 83375 galaxies created
--------------------------------
... all done
--------------------------------
Simulation completed in 10.87 seconds.


Kept 458324 of 2941105 objects in catalog
Occupying halos...
... 67102 of 458324 halos occupied
--------------------------------
Assigning values for centrals... 
... halo IDs, halo masses, positions
... luminosities


... Poisson satellite counts
... done
--------------------------------
Assigning values for satellites... 
... halo properties
... positions
... luminosities


... done
--------------------------------
... 83375 galaxies created
--------------------------------
... all done
--------------------------------
Simulation completed in 10.27 seconds.


In [7]:
class TwoPointCorrelationFunctionPeriodic(object):
    """ 2-point (cross-)correlation functions in bins of separation.
        Assumes complete, cubic periodic box throughout.
    """
    def __init__(self,lgbin=1,rmin=1e-2,rmax=3e1,nbin=15,Lbox=300.0,proj=0,pimax=60.0):
        self.lgbin = lgbin
        self.rmin = rmin
        self.rmax = rmax
        self.nbin = nbin
        self.Lbox = Lbox
        self.proj = proj
        self.pimax = pimax
        self.TINY = 1e-15
        self.N_SPLIT = 8000

        print_string = "Initialising 2pcf calculator for periodic boxes\n"
        print_string += "--------------------------------"
        print(print_string)

        if self.proj:
            print("... projected 2pcf calculated")
            if self.lgbin==1:
                self.rpbin = np.logspace(np.log10(self.rmin),np.log10(self.rmax),self.nbin+1)
                self.rpmid = np.sqrt(self.rpbin[1:]*self.rpbin[:-1])
            else:
                self.rpbin = np.linspace(self.rmin,self.rmax,self.nbin+1)
                self.rpmid = 0.5*(self.rpbin[1:]+self.rpbin[:-1])
            self.NR = 50
            self.NR_INT = self.NR*3
            self.RMIN = self.rmin
            self.RMAX = 0.5*self.Lbox
            self.rbin = np.logspace(np.log10(self.RMIN),np.log10(self.RMAX),self.NR+1)
            self.rmid = np.sqrt(self.rbin[1:]*self.rbin[:-1])
            self.rbin_int = np.logspace(np.log10(self.RMIN),np.log10(self.RMAX),self.NR_INT+1)
            self.rmid_int = np.sqrt(self.rbin_int[1:]*self.rbin_int[:-1])
            self.dlnr_int = np.log(self.rmid_int[1]/self.rmid_int[0])
        else:
            print("... monopole 2pcf calculated")
            if self.lgbin==1:
                self.rbin = np.logspace(np.log10(self.rmin),np.log10(self.rmax),self.nbin+1)
                self.rmid = np.sqrt(self.rbin[1:]*self.rbin[:-1])
            else:
                self.rbin = np.linspace(self.rmin,self.rmax,self.nbin+1)
                self.rmid = 0.5*(self.rbin[1:]+self.rbin[:-1])

        print_string = "... initialisation complete\n"
        print_string += "--------------------------------"
        print(print_string)

    def pair_counts(self,pos_1,pos_2):
        tree_1 = syspat.cKDTree(pos_1,boxsize=self.Lbox)
        tree_2 = syspat.cKDTree(pos_2,boxsize=self.Lbox)
        cum_counts = tree_1.count_neighbors(tree_2,self.rbin,cumulative=True)
        bin_counts = np.diff(cum_counts)
        del tree_1,tree_2
        gc.collect()
        return bin_counts

    def RR_theory(self):
        return 4*np.pi/3*(self.rbin[1:]**3-self.rbin[:-1]**3)/self.Lbox**3

    def twoPCF(self,pos_data,pos_data2=None):
        if pos_data.shape[1] != 3:
            raise TypeError("Incompatible data shape. Expected (ndata,3).")
        DD = self.DD_split(pos_data,pos_data2=pos_data2)
        RR = self.RR_theory()
        cf_r = DD/(RR + self.TINY) - 1.0

        if self.proj:
            print("... ... projecting")
            cf = np.zeros(self.nbin,dtype=float)
            cf_r_int = np.interp(self.rmid_int,self.rmid,cf_r)
            for rp in range(self.nbin):
                cond = (self.rmid_int > self.rpmid[rp]) & (self.rmid_int**2 < (self.rpmid[rp]**2 + self.pimax**2))
                ind = np.where(cond)[0]
                if ind.size:
                    Drp = self.rmid_int[ind[0]] - self.rpmid[rp]
                    add_this = np.sqrt(2*Drp/self.rpmid[rp])*cf_r_int[ind[0]]*self.rpmid[rp]
                    cf[rp] = add_this + 2*np.trapezoid((self.rmid_int[cond]*cf_r_int[cond]
                                                    /np.sqrt(self.rmid_int[cond]**2 - self.rpmid[rp]**2)),
                                                   x=self.rmid_int[cond])
        else:
            cf = 1.0*cf_r
        print("... ... done")
        return cf

    def DD_only(self,pos_data1,pos_data2=None):
        ndata1 = pos_data1.shape[0]
        if pos_data2 is None:
            cf = 1.0*self.pair_counts(pos_data1,pos_data1)/ndata1/(ndata1-1)
        else:
            ndata2 = pos_data2.shape[0]
            cf = 1.0*self.pair_counts(pos_data1,pos_data2)/ndata1/ndata2
        return cf

    def DD_split(self,pos_data,pos_data2=None):
        ndata = pos_data.shape[0]
        ndata_by_2 = ndata//2
        if ndata < self.N_SPLIT:
            print_string = '... ... calculating '
            print_string += "DD" if pos_data2 is None else "D1D2"
            print(print_string)
            cf = self.DD_only(pos_data,pos_data2=pos_data2)
        else:
            print("... ... binary split")
            if pos_data2 is None:
                pd1s = pos_data[:ndata_by_2].copy()
                ndata1s = pd1s.shape[0]
                pd1spr = pos_data[ndata_by_2:].copy()
                ndata1spr = pd1spr.shape[0]
                D1D1 = self.DD_split(pd1s)
                D1D1pr = self.DD_split(pd1s,pos_data2=pd1spr)
                D1prD1pr = self.DD_split(pd1spr)
                cf = ndata1s*(ndata1s-1)*D1D1 + 2*ndata1s*ndata1spr*D1D1pr + ndata1spr*(ndata1spr-1)*D1prD1pr
                cf = 1.0*cf/ndata/(ndata-1)
                del pd1s,pd1spr
                gc.collect()
            else:
                ndata2 = pos_data2.shape[0]
                ndata2_by_2 = ndata//2
                pd1s = pos_data[:ndata_by_2].copy()
                ndata1s = pd1s.shape[0]
                pd1spr = pos_data[ndata_by_2:].copy()
                ndata1spr = pd1spr.shape[0]
                pd2s = pos_data2[:ndata2_by_2].copy()
                ndata2s = pd2s.shape[0]
                pd2spr = pos_data2[ndata2_by_2:].copy()
                ndata2spr = pd2spr.shape[0]
                D1D2 = self.DD_split(pd1s,pos_data2=pd2s)
                D1D2pr = self.DD_split(pd1s,pos_data2=pd2spr)
                D1prD2 = self.DD_split(pd1spr,pos_data2=pd2s)
                D1prD2pr = self.DD_split(pd1spr,pos_data2=pd2spr)
                cf = ndata1s*ndata2s*D1D2 + ndata1s*ndata2spr*D1D2pr 
                cf = cf + ndata1spr*ndata2s*D1prD2 + ndata1spr*ndata2spr*D1prD2pr
                cf = 1.0*cf/ndata/ndata2
                del pd1s,pd1spr,pd2s,pd2spr
                gc.collect()
        return cf


In [ ]:
tpcf = TwoPointCorrelationFunctionPeriodic(rmin=0.1,rmax=40.,nbin=20,Lbox=mm.Lbox,
                                           proj=1,pimax=40.0)

cpos = np.array([centrals['x'],centrals['y'],centrals['z']]).T
spos = np.array([satellites['x'],satellites['y'],satellites['z']]).T
gpos = np.vstack((cpos,spos))

print('Centrals auto')
wp_cen = tpcf.twoPCF(cpos,cpos)
print('Satellites auto')
wp_sat = tpcf.twoPCF(spos,spos)
print('All auto')
wp_gal = tpcf.twoPCF(gpos,gpos)

plt.figure(figsize=(8,6))
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r"$r_{\rm p} \, (h^{-1}{\rm Mpc})$")
plt.ylabel(r"$w_{\rm p}(r_{\rm p}) \, (h^{-1}{\rm Mpc})$")
plt.plot(tpcf.rpmid,wp_cen,'b-',label='centrals')
plt.plot(tpcf.rpmid,wp_sat,'g-',label='satellites')
plt.plot(tpcf.rpmid,wp_gal,'k-',label='cen + sat (vectorized)')
plt.title('Vectorized Mocker - Projected 2PCF')
plt.legend()
plt.tight_layout()
plt.show()


Initialising 2pcf calculator for periodic boxes
--------------------------------
... projected 2pcf calculated
... initialisation complete
--------------------------------
Centrals auto
... ... binary split
... ... binary split
... ... binary split
... ... binary split
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... binary split
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... binary split
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... binary split
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... binary split
... ... binary split
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... binary split
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... ... calculating D1D2
... 